# Filterpass — SAP Training Pipeline
Wav2Vec2-base fine-tuned with a Self-Attention Pooling head for deepfake audio detection.

**Before running:**
1. `Runtime → Change runtime type → GPU` (T4 minimum, A100 recommended)
2. Upload your `data_training/` folder to Google Drive under `MyDrive/filterpass/`
3. Run all cells top to bottom

## 1. Install Dependencies

In [ ]:
!pip install -q transformers torchaudio librosa scikit-learn matplotlib seaborn soundfile tqdm

## 2. Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Adjust these if your Drive layout differs ─────────────────────────────────
BASE_DIR        = '/content/drive/MyDrive/filterpass/data_training'
CHECKPOINT_DIR  = '/content/drive/MyDrive/filterpass/checkpoints'
CHECKPOINT_NAME = 'best_model_SAP.pt'
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Base dir   : {BASE_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## 3. Config

In [ ]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}  ({vram_gb:.1f} GB VRAM)')
else:
    print('No GPU — switch runtime type before proceeding.')

# ── Tune batch_size to your GPU ──────────────────────────────────────────────
# T4  (16 GB): batch_size=32
# A100(40 GB): batch_size=64  (disable gradient_checkpointing too)
CONFIG = {
    'batch_size':             32,
    'max_epochs':             25,
    'patience':               5,
    'min_delta':              0.001,   # 0.1pp EER — gains below this are marginal
    'grad_accum_steps':       1,
    'lr_encoder':             1e-6,
    'lr_classifier':          1e-4,
    'weight_decay':           0.01,
    'warmup_ratio':           0.1,
    'max_grad_norm':          0.5,
    'class_weights':          [1.0, 1.0],
    'num_workers':            4,       # Linux multiprocessing — no Windows spawn overhead
    'prefetch_factor':        4,
    'freeze_encoder_layers':  6,       # freeze 0–5; train 6–11  (A100: try 4 or 0)
    'gradient_checkpointing': True,    # set False on A100 — enough VRAM
    'wav2vec2_model':         'facebook/wav2vec2-base',
}

## 4. Model Definition

In [ ]:
import torch.nn as nn
from transformers import Wav2Vec2Model


class SelfAttentionPooling(nn.Module):
    """Learns a per-frame attention weight; collapses (B, T, D) → (B, D)."""

    def __init__(self, input_dim: int):
        super().__init__()
        self.W    = nn.Linear(input_dim, 256)
        self.tanh = nn.Tanh()
        self.V    = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_weights = torch.softmax(self.V(self.tanh(self.W(x))), dim=1)
        return torch.sum(x * attn_weights, dim=1)


class SAPClassifier(nn.Module):
    """Wav2Vec2-base + Self-Attention Pooling classification head."""

    def __init__(
        self,
        model_name:       str  = 'facebook/wav2vec2-base',
        freeze_extractor: bool = True,
    ):
        super().__init__()
        print('Initialising SAPClassifier')
        self.encoder = Wav2Vec2Model.from_pretrained(model_name)

        if freeze_extractor:
            for param in self.encoder.feature_extractor.parameters():
                param.requires_grad = False

        self.attention_pooling = SelfAttentionPooling(input_dim=768)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2),
        )

    def forward(
        self,
        input_values:   torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        hidden = self.encoder(input_values, attention_mask=attention_mask).last_hidden_state
        pooled = self.attention_pooling(hidden)
        return self.classifier(pooled)

## 5. Dataset

In [ ]:
import numpy as np
import librosa
from torch.utils.data import Dataset
from tqdm import tqdm

MAX_SAMPLES = 64_000  # 4s at 16 kHz

_SPLIT_MAP = {
    'train': (
        'augmented/train/flac',
        'augmented/keys/ASVspoof2019.LA.cm.train.augmented.txt',
    ),
    'dev': (
        'ASVspoof2019_LA_dev/flac',
        'keys/ASVspoof2019.LA.cm.dev.trl.txt',
    ),
    'eval': (
        'ASVspoof2019_LA_eval/flac',
        'keys/ASVspoof2019.LA.cm.eval.trl.txt',
    ),
}
_LABEL_MAP = {'bonafide': 0, 'spoof': 1}


def _read_audio(path: str) -> tuple[np.ndarray, int]:
    try:
        import torchaudio
        waveform, sr = torchaudio.load(path, backend='ffmpeg')
        return waveform.squeeze(0).numpy(), sr
    except Exception:
        return librosa.load(path, sr=None, mono=False)


def _preprocess(waveform: np.ndarray, sr: int) -> np.ndarray:
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=0)
    if sr != 16_000:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=16_000)
    max_val = np.max(np.abs(waveform))
    if max_val > 0:
        waveform = waveform / max_val
    if len(waveform) >= MAX_SAMPLES:
        waveform = waveform[:MAX_SAMPLES]
    else:
        waveform = np.pad(waveform, (0, MAX_SAMPLES - len(waveform)))
    return waveform.astype(np.float32)


class ASVspoofDataset(Dataset):
    """
    4-second fixed-length dataset for ASVspoof 2019 LA.
    Loads from .npy cache when available; falls back to live FLAC decoding.
    """

    def __init__(self, base_dir: str, split: str = 'train'):
        audio_subdir, protocol_subpath = _SPLIT_MAP[split]
        self.audio_dir = os.path.join(base_dir, audio_subdir)
        protocol_path  = os.path.join(base_dir, protocol_subpath)

        cache_dir = os.path.join(base_dir, 'cache', split)
        self.cache_dir = cache_dir if os.path.isdir(cache_dir) else None

        self.samples: list[tuple[str, int]] = []
        skipped = 0
        with open(protocol_path) as f:
            for line in tqdm(f, desc=f'Indexing {split}'):
                parts = line.strip().split()
                if len(parts) < 5:
                    skipped += 1
                    continue
                utt_id = parts[1]
                label  = _LABEL_MAP.get(parts[4], -1)
                self.samples.append((
                    os.path.join(self.audio_dir, f'{utt_id}.flac'), label
                ))

        cache_status = f'cache at {cache_dir}' if self.cache_dir else 'no cache'
        print(f'{split}: {len(self.samples)} utterances  |  {skipped} skipped  |  {cache_status}')

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        path, label = self.samples[idx]
        return {
            'input_values': torch.from_numpy(self._load(path)),
            'labels':       torch.tensor(label, dtype=torch.long),
        }

    def _load(self, path: str) -> np.ndarray:
        if self.cache_dir is not None:
            cache_path = os.path.join(self.cache_dir, os.path.splitext(os.path.basename(path))[0] + '.npy')
            if os.path.exists(cache_path):
                return np.load(cache_path)
        try:
            return _preprocess(*_read_audio(path))
        except Exception as e:
            tqdm.write(f'[ERROR] {os.path.basename(path)}: {e}')
            return np.zeros(MAX_SAMPLES, dtype=np.float32)

## 6. Build .npy Cache (run once)
Decodes every FLAC to a pre-processed float32 array. Subsequent training epochs load `.npy` directly — eliminates FLAC decoding overhead.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed


def _cache_file(args: tuple) -> tuple[bool, str]:
    flac_path, cache_path = args
    if os.path.exists(cache_path):
        return True, ''
    try:
        np.save(cache_path, _preprocess(*_read_audio(flac_path)))
        return True, ''
    except Exception as e:
        return False, f'{flac_path}: {e}'


def build_cache(
    base_dir: str,
    splits: tuple[str, ...] = ('train', 'dev'),
    max_workers: int = 8,
) -> None:
    for split in splits:
        audio_subdir, protocol_subpath = _SPLIT_MAP[split]
        audio_dir     = os.path.join(base_dir, audio_subdir)
        protocol_path = os.path.join(base_dir, protocol_subpath)
        cache_dir     = os.path.join(base_dir, 'cache', split)
        os.makedirs(cache_dir, exist_ok=True)

        with open(protocol_path) as f:
            lines = [ln.strip().split() for ln in f if ln.strip()]

        tasks = [
            (
                os.path.join(audio_dir,  f'{p[1]}.flac'),
                os.path.join(cache_dir,  f'{p[1]}.npy'),
            )
            for p in lines if len(p) >= 5
        ]

        already = sum(1 for _, cp in tasks if os.path.exists(cp))
        print(f'[{split}] {len(tasks)} total  |  {already} cached  |  {len(tasks)-already} remaining')
        if already == len(tasks):
            continue

        errors = 0
        with ProcessPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(_cache_file, t): t for t in tasks}
            with tqdm(total=len(tasks), desc=f'Caching {split}') as pbar:
                for future in as_completed(futures):
                    ok, msg = future.result()
                    if not ok:
                        errors += 1
                        tqdm.write(f'[ERROR] {msg}')
                    pbar.update(1)
        print(f'[{split}] Done  |  {errors} errors')


build_cache(BASE_DIR)

## 7. Evaluation Utilities

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, auc, confusion_matrix,
    f1_score, precision_score, recall_score, roc_curve,
)


def compute_eer(
    labels: np.ndarray, scores: np.ndarray
) -> tuple[np.ndarray, np.ndarray, int, float]:
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    return fpr, tpr, idx, (fpr[idx] + fnr[idx]) / 2


def run_inference(
    model, loader, device, loss_fn=None
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float | None]:
    model.eval()
    all_labels, all_scores, all_preds = [], [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            input_values = batch['input_values'].to(device)
            labels       = batch['labels'].to(device)

            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(input_values)
                if loss_fn is not None:
                    total_loss += loss_fn(logits, labels).item()

            probs = torch.softmax(logits.float(), dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_scores.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader) if loss_fn is not None else None
    return np.array(all_labels), np.array(all_scores), np.array(all_preds), avg_loss


def plot_training_history(history: dict, save_path: str | None = None) -> None:
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs, history['train_loss'], label='Train Loss')
    axes[0].plot(epochs, history['val_loss'],   label='Val Loss')
    axes[0].set(xlabel='Epoch', ylabel='Loss', title='Train vs Val Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], label='Train Acc')
    axes[1].plot(epochs, history['val_acc'],   label='Val Acc')
    axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Train vs Val Accuracy')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, [e * 100 for e in history['val_eer']], color='red', label='Val EER (%)')
    axes[2].set(xlabel='Epoch', ylabel='EER (%)', title='Validation EER')
    axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_results(
    labels: np.ndarray,
    scores: np.ndarray,
    preds: np.ndarray,
    save_path: str | None = None,
) -> None:
    fpr, tpr, eer_idx, eer = compute_eer(labels, scores)
    roc_auc = auc(fpr, tpr)

    print(f'EER:       {eer*100:.2f}%')
    print(f'Accuracy:  {accuracy_score(labels, preds)*100:.2f}%')
    print(f'F1:        {f1_score(labels, preds, average="weighted"):.4f}')
    print(f'AUC:       {roc_auc:.4f}')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={roc_auc:.3f}')
    ax1.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--')
    ax1.plot(fpr[eer_idx], tpr[eer_idx], 'ro', markersize=8, label=f'EER={eer*100:.2f}%')
    ax1.set(xlabel='FPR', ylabel='TPR', title='ROC Curve')
    ax1.legend(); ax1.grid(alpha=0.3)

    sns.heatmap(
        confusion_matrix(labels, preds), annot=True, fmt='d', cmap='Blues', ax=ax2,
        xticklabels=['Bonafide', 'Spoof'], yticklabels=['Bonafide', 'Spoof'],
    )
    ax2.set(xlabel='Predicted', ylabel='True', title='Confusion Matrix')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

## 8. Training

In [ ]:
import random
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup


def configure_cuda() -> None:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


configure_cuda()
set_seed(SEED)

In [ ]:
nw = CONFIG['num_workers']
pf = CONFIG['prefetch_factor']

train_dataset = ASVspoofDataset(BASE_DIR, split='train')
dev_dataset   = ASVspoofDataset(BASE_DIR, split='dev')

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=nw, pin_memory=True,
    prefetch_factor=pf, persistent_workers=True,
)
dev_loader = DataLoader(
    dev_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=nw, pin_memory=True,
    prefetch_factor=pf, persistent_workers=True,
)

print(f'Train batches: {len(train_loader)}  |  Dev batches: {len(dev_loader)}')

In [ ]:
model = SAPClassifier(CONFIG['wav2vec2_model']).to(DEVICE)

# Freeze bottom N transformer blocks
n = CONFIG['freeze_encoder_layers']
if n > 0:
    layers = model.encoder.encoder.layers
    for layer in layers[:n]:
        for param in layer.parameters():
            param.requires_grad = False
    print(f'Frozen Wav2Vec2 layers 0–{n-1} of {len(layers)}')

if CONFIG['gradient_checkpointing']:
    model.encoder.gradient_checkpointing_enable()
    print('Gradient checkpointing enabled')

optimizer = torch.optim.AdamW(
    [
        {'params': filter(lambda p: p.requires_grad, model.encoder.parameters()), 'lr': CONFIG['lr_encoder']},
        {'params': model.classifier.parameters(), 'lr': CONFIG['lr_classifier']},
    ],
    weight_decay=CONFIG['weight_decay'],
)

steps_per_epoch = len(train_loader) // CONFIG['grad_accum_steps']
total_steps     = steps_per_epoch * CONFIG['max_epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * CONFIG['warmup_ratio']),
    num_training_steps=total_steps,
)

class_weights = torch.tensor(CONFIG['class_weights']).to(DEVICE)
loss_fn       = nn.CrossEntropyLoss(weight=class_weights)
scaler        = torch.amp.GradScaler('cuda')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

In [ ]:
from tqdm.notebook import tqdm as tqdm_nb

checkpoint_path   = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAME)
history_plot_path = os.path.join(CHECKPOINT_DIR, 'training_history.png')

best_eer        = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_eer': []}

accum_steps = CONFIG['grad_accum_steps']

for epoch in range(CONFIG['max_epochs']):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm_nb(train_loader, desc=f'Epoch {epoch+1}/{CONFIG["max_epochs"]}')
    optimizer.zero_grad()

    for step, batch in enumerate(pbar):
        input_values = batch['input_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(input_values)
            loss   = loss_fn(logits, labels) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)
        pbar.set_postfix({'loss': f'{loss.item()*accum_steps:.4f}'})

    avg_loss  = total_loss / len(train_loader)
    train_acc = correct / total
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(train_acc)

    labels_dev, scores_dev, preds_dev, val_loss = run_inference(model, dev_loader, DEVICE, loss_fn)
    history['val_loss'].append(val_loss)

    dev_eer = compute_eer(labels_dev, scores_dev)[3]
    val_acc = accuracy_score(labels_dev, preds_dev)
    history['val_acc'].append(val_acc)
    history['val_eer'].append(dev_eer)

    print(
        f'Epoch {epoch+1} | Loss: {avg_loss:.4f} | '
        f'Train Acc: {train_acc*100:.2f}% | '
        f'Val Acc: {val_acc*100:.2f}% | '
        f'Dev EER: {dev_eer*100:.2f}%'
    )

    improvement = best_eer - dev_eer
    if dev_eer < best_eer:
        best_eer = dev_eer
        torch.save(model.state_dict(), checkpoint_path)
        print(f'  New best EER — saved to {checkpoint_path}')

    if improvement >= CONFIG['min_delta']:
        patience_counter = 0
    else:
        patience_counter += 1
        print(f'  Marginal gain ({improvement*100:.3f}pp). Patience: {patience_counter}/{CONFIG["patience"]}')
        if patience_counter >= CONFIG['patience']:
            print(f'  Early stopping at epoch {epoch+1} — gains below min_delta.')
            break

    plot_training_history(history, save_path=history_plot_path)

print(f'\nTraining complete. Best EER: {best_eer*100:.2f}%')

## 9. Evaluate on Eval Set
Loads the best checkpoint and runs evaluation on the held-out ASVspoof 2019 LA eval partition.

In [ ]:
eval_dataset = ASVspoofDataset(BASE_DIR, split='eval')
eval_loader  = DataLoader(
    eval_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=nw, pin_memory=True,
    prefetch_factor=pf, persistent_workers=True,
)

eval_model = SAPClassifier(CONFIG['wav2vec2_model']).to(DEVICE)
eval_model.load_state_dict(
    torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
)

labels_eval, scores_eval, preds_eval, _ = run_inference(eval_model, eval_loader, DEVICE)
plot_results(
    labels_eval, scores_eval, preds_eval,
    save_path=os.path.join(CHECKPOINT_DIR, 'eval_results.png'),
)